# Getting Started: A First Steady Simulation

This tutorial walks through a first steady aerodynamic simulation with Ptera
Software. You will build a simple airplane from scratch, one object at a time:

1. Airfoil
2. WingCrossSection
3. Wing
4. Airplane
5. OperatingPoint
6. SteadyProblem
7. Solver

By the end you will have solved for the lift and induced drag of a rectangular
wing and seen how to log, save, and plot the results.

This tutorial uses the steady horseshoe vortex lattice method (VLM). Ptera
Software also ships steady ring VLM, unsteady ring VLM (UVLM), aeroelastic, and
free flight solvers; the object model you learn here is shared by all of them.


## Install

Install the package in your environment if you have not already:

```shell
pip install pterasoftware
```

Then import the package. The examples in this tutorial assume you are working
in a Jupyter notebook or an interactive Python session.


In [ ]:
import pterasoftware as ps

## 1. Airfoil

An `Airfoil` defines the two-dimensional shape of a wing cross section. Ptera
Software ships a large database of airfoils (courtesy of the UIUC Airfoil
Coordinates Database); you select one by name.

Here we use the classic NACA 2412 airfoil:


In [ ]:
airfoil = ps.geometry.airfoil.Airfoil(name="naca2412")
airfoil

## 2. WingCrossSection

A `WingCrossSection` places an `Airfoil` at a location along the wing and
controls how that station is discretized into panels. `num_spanwise_panels`
controls the panel resolution in the spanwise direction, and `chord` sets the
local chord length.

A wing is built from two or more cross sections. Here we define a root station
and a tip station. The tip is offset from the root with `Lp_Wcsp_Lpp`: the
second component is the spanwise offset, the first is the streamwise offset
(sweep), and the third is the vertical offset (dihedral).


In [ ]:
root_cross_section = ps.geometry.wing_cross_section.WingCrossSection(
    airfoil=airfoil,
    num_spanwise_panels=8,
    chord=1.75,
    Lp_Wcsp_Lpp=(0.0, 0.0, 0.0),
    control_surface_symmetry_type="symmetric",
)
root_cross_section

In [ ]:
tip_cross_section = ps.geometry.wing_cross_section.WingCrossSection(
    airfoil=airfoil,
    num_spanwise_panels=None,
    chord=1.5,
    Lp_Wcsp_Lpp=(0.75, 6.0, 1.0),
    control_surface_symmetry_type="symmetric",
)
tip_cross_section

## 3. Wing

A `Wing` groups the cross sections into a lifting surface and discretizes it
into panels in the chordwise direction (`num_chordwise_panels`).

Setting `symmetric=True` mirrors the wing about the plane defined by
`symmetryNormal_G` and `symmetryPoint_G_Cg`, which models the starboard half of
a conventional airplane. The cross sections you defined above describe only the
right half of the wing; the solver creates the mirrored left half for you.


In [ ]:
wing = ps.geometry.wing.Wing(
    wing_cross_sections=[root_cross_section, tip_cross_section],
    symmetric=True,
    symmetryNormal_G=(0.0, 1.0, 0.0),
    symmetryPoint_G_Cg=(0.0, 0.0, 0.0),
    num_chordwise_panels=6,
)
wing

## 4. Airplane

An `Airplane` is a collection of one or more `Wing` objects. This is the top
level of the geometry hierarchy.


In [ ]:
airplane = ps.geometry.airplane.Airplane(
    wings=[wing],
)
airplane

## 5. OperatingPoint

An `OperatingPoint` describes the flight condition: free stream velocity,
angle of attack, sideslip, air density, and more. The default values below are
a typical low-speed cruise condition.


In [ ]:
operating_point = ps.operating_point.OperatingPoint()
operating_point

## 6. SteadyProblem

A `SteadyProblem` bundles the geometry and the operating point into a single
object that the solver can act on.


In [ ]:
problem = ps.problems.SteadyProblem(
    airplanes=[airplane],
    operating_point=operating_point,
)
problem

## 7. SteadyHorseshoeVortexLatticeMethodSolver

The `SteadyHorseshoeVortexLatticeMethodSolver` solves the problem. Its `run`
method builds the vortex lattice and solves the linear system, which takes less
than a second for this geometry thanks to JIT compilation and parallelization.


In [ ]:
solver = (
    ps.steady_horseshoe_vortex_lattice_method.SteadyHorseshoeVortexLatticeMethodSolver(
        steady_problem=problem,
    )
)
solver.run()
solver

## 8. Results

The solver stores the converged results directly on the `Airplane` object.
Common quantities are the total lift and induced drag coefficients:


In [ ]:
print("Lift coefficient:", airplane.liftCoefficient_W)
print("Induced drag coefficient:", airplane.inducedDragCoefficient_W)
print("Force vector (N):", airplane.forces_W)
print("Force coefficient vector:", airplane.forceCoefficients_W)

Per-panel data is available on `solver.panels`, which is useful for
post-processing such as spanwise load plots.


In [ ]:
print("Number of panels:", len(solver.panels))
print("Panel object:", solver.panels[0])

## 9. Visualization

The `output` module can draw the solved airplane and its aerodynamic loads.
Ptera Software renders 3D scenes with PyVista; in a notebook you can save the
scene to an image with `save=True`.

Here we plot the spanwise lift distribution with Matplotlib to keep the
tutorial self-contained and fast:


In [ ]:
import matplotlib.pyplot as plt

# Pull per-panel lift forces and the spanwise location of each panel.
spanwise_locations = []
lift_forces = []
for panel in solver.panels:
    # Panel corner points in global coordinates (G) about the center of
    # gravity (Cg). The second component is the spanwise direction.
    y = float(panel.Brpp_G_Cg[1])
    spanwise_locations.append(y)
    lift_forces.append(float(panel.forces_W[2]))

plt.figure(figsize=(7.0, 4.0))
plt.plot(spanwise_locations, lift_forces, ".-")
plt.xlabel("Spanwise location (m)")
plt.ylabel("Lift force (N)")
plt.title("Spanwise lift distribution")
plt.grid(True)
plt.show()

## 10. Logging

The `output` module can write a formatted log of the simulation setup and
results. Call `ps.set_up_logging` once to configure logging, then use
`ps.output.log_results`:


In [ ]:
import logging

ps.set_up_logging(level="Info")
ps.output.log_results(solver)

## 11. Saving and loading

Solved simulations can be serialized to JSON and reloaded without re-running,
which is handy for sharing or post-processing later.


In [ ]:
ps.save("getting_started_solution.json", solver)
loaded = ps.load("getting_started_solution.json")
print("Lift coefficient after reload:", loaded.airplanes[0].liftCoefficient_W)

## Summary

You have now run your first Ptera Software simulation. The object model is
consistent across all five solvers, so the same pattern applies to steady ring
VLM, unsteady ring VLM, aeroelastic, and free flight simulations:

```text
Airfoil -> WingCrossSection -> Wing -> Airplane
OperatingPoint -> Problem -> Solver -> run()
```

To go further, look at the example scripts in `examples/` and the API
reference on the documentation site.
